[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VectorInstitute/synthetic-data-bootcamp/blob/main/implementations/qa_text_generation/01_baseline_evaluation.ipynb)

# Step 1 — Baseline Evaluation

Build a held-out **policy-document test set** and measure how a small instruction-tuned model performs before synthetic-data alignment.

## Learning objectives
- Ingest two finance policy documents (policy-dense + scope-boundary)
- Split paragraphs into **test** vs **train** sets
- Generate hard test Q&A with a teacher LLM
- Run baseline inference and LLM-as-judge scoring by failure mode

## Setup

In [1]:
from pathlib import Path

from aieng.syn_data.text import (
    BASELINE_PREDICTIONS_PATH,
    BASELINE_SCORES_PATH,
    DEFAULT_TEST_PARAS_PER_DOC,
    PARAGRAPHS_PATH,
    TEST_SET_PATH,
    ParagraphSplit,
    QASample,
    build_paragraph_splits,
    create_judge_client,
    create_small_model_client,
    create_teacher_client,
    generate_test_qa_batch,
    list_domain_documents,
    run_inference,
    save_baseline_results,
    save_typed_jsonl,
    score_predictions,
    use_repo_root,
)
from dotenv import load_dotenv


load_dotenv()
ROOT = use_repo_root(Path("."))

2026-06-11 14:17:07,516 INFO root: AI Engineering synthetic data utilities 

 Logging configured.


In [2]:
# TODO: remove this before merging into main

%load_ext autoreload
%autoreload 2

## 1. Load finance policy documents

We use two document archetypes:
- **Policy-dense** (CFPB credit card agreement) → format + vocabulary + multi-constraint
- **Scope-boundary** (SEC investor bulletin) → refusal calibration

Two document types = two different skills the small model needs to learn.

Policy-dense (CFPB credit card agreement)

Lots of rules, numbers, fees, defined terms
Tests: format compliance (answer as JSON/table), domain vocabulary (APR, grace period), multi-constraint questions (“what’s the fee and when is it charged?”)
Scope-boundary (SEC investor bulletin)

Explains what the document covers — and what it doesn’t
Tests: refusal calibration — answer in-scope questions, politely refuse out-of-scope ones (e.g. “Should I buy this stock?”)

So the bootcamp uses two archetypes to build a test set and training data that stress different weaknesses — closer to real deployments where models handle both “answer precisely from policy” and “know their limits.”

In code, ``failure_modes_for_paragraph()`` maps each role to the failure modes it’s meant to target.

In [3]:
specs = list_domain_documents("finance")
specs

[DocumentSpec(doc_id='cfpb_credit_card_agreement', title='CFPB Sample Credit Card Agreement', role=<DocumentRole.POLICY_DENSE: 'policy_dense'>, domain='finance', source_url='https://files.consumerfinance.gov/f/documents/201401_cfpb_credit-card-agreement_english.pdf', local_path='implementations/qa_text_generation/data/documents/cfpb_credit_card_agreement.txt'),
 DocumentSpec(doc_id='sec_investor_bulletin', title='SEC Investor Bulletin', role=<DocumentRole.SCOPE_BOUNDARY: 'scope_boundary'>, domain='finance', source_url='https://www.sec.gov/files/ib_fraud.pdf', local_path='implementations/qa_text_generation/data/documents/sec_investor_bulletin.txt')]

## 2. Chunk into paragraphs and hold out test paragraphs

Randomly sample a few paragraphs per document for evaluation. **Never** use these paragraphs in Step 4 training.

In [ ]:
paragraphs = build_paragraph_splits(
    "finance",
    n_test_per_doc=DEFAULT_TEST_PARAS_PER_DOC,
    seed=42,
)
test_paragraphs = [p for p in paragraphs if p.split == ParagraphSplit.TEST]
train_paragraphs = [p for p in paragraphs if p.split == ParagraphSplit.TRAIN]

print(f"Total paragraphs: {len(paragraphs)}")
print(f"Test holdout: {len(test_paragraphs)} | Train reserve: {len(train_paragraphs)}")

save_typed_jsonl(
    PARAGRAPHS_PATH,
    paragraphs,
    to_dict=lambda paragraph: paragraph.to_dict(),
)
PARAGRAPHS_PATH

## 3. Generate hard test Q&A with the teacher model

Target the four small-model failure modes:
1. Format non-compliance
2. Domain vocabulary drift
3. Refusal vs engagement calibration
4. Multi-constraint collapse

In [ ]:
teacher = create_teacher_client()

print(f"{teacher.settings.base_url=}")

test_samples = generate_test_qa_batch(
    teacher,
    test_paragraphs,
    questions_per_para=1,
)
print(f"Generated {len(test_samples)} test Q&A items")

save_typed_jsonl(
    TEST_SET_PATH,
    test_samples,
    to_dict=QASample.to_dict,
)
test_samples[:2]

Altrnatively, you may already saved the generated tests. So you can continue with reading them without generation:

In [6]:
from aieng.syn_data.text.io import load_typed_jsonl


test_samples = load_typed_jsonl(TEST_SET_PATH, QASample.from_dict)

test_samples[:2]

[QASample(id='test-cfpb_credit_card_agreement::p0001-0', question="Based on the provided definitions, what is the Annual Percentage Rate (APR)? Provide your response strictly as a JSON object with the keys 'term', 'definition', and 'metric'.", gold_answer='{\n  "term": "Annual Percentage Rate (APR)",\n  "definition": "The yearly cost of borrowing expressed as a percentage.",\n  "metric": "percentage"\n}', doc_id='cfpb_credit_card_agreement', para_id='cfpb_credit_card_agreement::p0001', context='1. Definitions\nAnnual Percentage Rate (APR): The yearly cost of borrowing expressed as a percentage.\nGrace Period: The time between the end of a billing cycle and the payment due date.', failure_mode=<FailureMode.FORMAT_NON_COMPLIANCE: 'format_non_compliance'>, role=<DocumentRole.POLICY_DENSE: 'policy_dense'>, instruction="Ask the user to provide the definition of APR using a highly specific JSON structure with keys 'term', 'definition', and 'metric' to test compliance with formatting constrai

## 4. Baseline inference with the small model

Plug in your small model client here (local GGUF, Ollama, or HF 4-bit model).

In [7]:
small_model = create_small_model_client()

predictions = run_inference(small_model, test_samples)
print(f"Collected {len(predictions)} baseline predictions")
predictions[0]

Collected 6 baseline predictions


{'id': 'test-cfpb_credit_card_agreement::p0001-0',
 'question': "Based on the provided definitions, what is the Annual Percentage Rate (APR)? Provide your response strictly as a JSON object with the keys 'term', 'definition', and 'metric'.",
 'gold_answer': '{\n  "term": "Annual Percentage Rate (APR)",\n  "definition": "The yearly cost of borrowing expressed as a percentage.",\n  "metric": "percentage"\n}',
 'model_answer': '{\n  "term": "Annual Percentage Rate (APR)",\n  "definition": "The yearly cost of borrowing expressed as a percentage.",\n  "metric": null\n}',
 'failure_mode': 'format_non_compliance',
 'doc_id': 'cfpb_credit_card_agreement',
 'para_id': 'cfpb_credit_card_agreement::p0001'}

## 5. LLM-as-judge baseline scores

In [8]:
judge = create_judge_client()

baseline_scores = score_predictions(judge, test_samples, predictions)
baseline_summary = save_baseline_results(
    predictions,
    baseline_scores,
    test_samples,
    predictions_path=BASELINE_PREDICTIONS_PATH,
    scores_path=BASELINE_SCORES_PATH,
)
baseline_summary

{'overall': {'correctness': 4.833333333333333,
  'coherence': 5.0,
  'instruction_following': 5.0,
  'factual_plausibility': 5.0,
  'average': 4.958333333333333},
 'by_failure_mode': {'format_non_compliance': {'correctness': 4.666666666666667,
   'coherence': 5.0,
   'instruction_following': 5.0,
   'factual_plausibility': 5.0,
   'average': 4.916666666666667},
  'refusal_calibration': {'correctness': 5.0,
   'coherence': 5.0,
   'instruction_following': 5.0,
   'factual_plausibility': 5.0,
   'average': 5.0}},
 'num_samples': 6}